[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/02_databases/02_db_explore.ipynb)


# 02 — 항체 DB landscape (RCSB 스냅샷 직접 조회)

> 본문: [`02_databases.md`](02_databases.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 6초** (실측 — 도구가 도는 시간만. clone·pip·apt 설치 시간은 빼고 잰 값이에요)
> 코랩 무료 런타임은 코어가 2개뿐이라 설치에 1~2분, 무거운 예측 단계는 몇 배까지 더 걸릴 수 있어요 — 정상입니다.

**이 노트북은 도구를 직접 돌립니다.** RCSB Search/Data API 로 항체–항원 복합체 스냅샷을 **직접 만들어** `my_run/` 에 저장해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "02_databases"
PIP_PKGS = "pandas requests"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = False    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

QUIET_WARNINGS = True   # 라이브러리 내부 경고 소음을 끕니다. 다 보고 싶으면 False 로 두세요

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요.

    igfold·biopython·transformers 는 **자기 패키지 소스 줄번호**가 찍힌
    DeprecationWarning/FutureWarning 을 실제 결과보다 길게 쏟아내요. 그게 성공 메시지를
    덮어버려 실패로 오해하게 만들어서, 기본으로 PYTHONWARNINGS=ignore 를 걸어 둡니다.
    도구가 직접 print 하는 안내·에러는 그대로 남아요(warnings 모듈만 막는 거예요)."""
    args = [str(a) for a in args]
    # 절대경로를 그대로 찍으면 한 줄이 화면을 넘겨 정작 중요한 옵션이 안 보여요.
    # /usr/bin/python3 → python, /…/scripts/x.py → scripts/x.py 로 줄여서 보여줍니다.
    def _short(s):
        if s == PY:
            return "python"
        return "scripts/" + s[len(str(SCRIPTS)) + 1:] if s.startswith(str(SCRIPTS) + os.sep) else s
    print("$", " ".join(_short(a) for a in args))
    env = {**os.environ, "PYTHONWARNINGS": "ignore"} if QUIET_WARNINGS else None
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout, env=env)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 먼저 지도 — DB 성격별 분류와 확인 필드 (본문 2.1~2.6)

항체 DB 는 성격이 제각각이라 "어디서 무엇을 가져오나"부터 정해야 해요. 성격으로 나눈 여섯 유형을
세워 두고, 그중 **구조 DB** 축을 2절에서 직접 뽑아 봅니다. 아래 표의 여섯 줄이 본문 2.2~2.6 을
한 눈에 접은 것이라, 뒤 챕터에서 어느 DB 를 다시 만나는지도 마지막 열로 확인할 수 있어요.

구조 DB 를 볼 때 확인해야 할 필드 목록(본문 2.2)도 같이 만들어 둬요 — 3절에서 이 목록이
RCSB 스냅샷으로 **몇 개나 채워지는지** 세어 볼 거예요.

In [ ]:
import pandas as pd

dbs = pd.DataFrame([
    ["구조 DB",             "SAbDab / SAbDab-nano / IMGT-3Dstructure-DB", "항체 구조·항원-항체 복합체", "구조 비교·epitope/paratope", "Ch.06·07"],
    ["서열 repertoire DB",  "OAS / AIRR Data Commons / iReceptor",        "BCR·항체 대량 서열",        "naturalness·germline",      "Ch.09"],
    ["치료용 항체 DB",       "Thera-SAbDab",                              "임상·승인 항체 서열·메타",   "benchmark·developability",  "Ch.05"],
    ["질병·항원 특화 DB",    "CoV-AbDab / IEDB",                          "항원 특이 항체·epitope",     "중화항체·epitope 분석",      "Ch.07"],
    ["affinity/mutation DB","AB-Bind / SKEMPI 2.0",                      "mutation별 binding 변화",   "affinity maturation·ΔΔG",   "-"],
    ["통합 분석 시스템",     "abYsis",                                    "sequence/structure annotation", "항체-aware annotation", "-"],
], columns=["DB 유형", "대표 DB", "주요 데이터", "주 용도", "이 과정에서"])
display(dbs)

# 본문 2.2 의 SAbDab 확인 필드 → 이 스냅샷의 어느 컬럼이 그 역할을 하는지(없으면 None)
SABDAB_FIELDS = {
    "PDB ID":               "pdb_id",
    "heavy/light chain ID": "heavy_chains",
    "antigen chain ID":     "antigen_chains",
    "resolution":           "resolution_A",
    "antigen type":         "antigen_name",
    "antibody species":     None,
    "affinity value":       None,
    "bound/unbound":        None,
    "CDR sequence/length":  None,
}
print(f"\n구조 DB 에서 확인할 필드 {len(SABDAB_FIELDS)}개 —", " · ".join(SABDAB_FIELDS))
print(f"판정 — {len(dbs)}개 유형 중 다음 절에서 직접 뽑는 것은 첫 줄의 구조 DB 축이에요.")

## 2) 직접 실행 — RCSB 에서 항체-항원 복합체 스냅샷 만들기 (본문 2.2b)

SAbDab·Thera-SAbDab 웹 UI 는 스크립트로 바로 긁기 어려워요(JS 렌더링 앱이라 HTML 만 돌아옵니다).
그래서 **같은 원본인 PDB** 를 RCSB **Search API + Data API** 로 직접 조회해 "SAbDab스러운" 요약 표를 만듭니다.

```bash
python scripts/fetch_rcsb_ab_snapshot.py --rows 12 --out my_run/rcsb_ab_complexes.csv
```
- 검색 조건 — X-ray · 해상도 ≤ 2.5 Å · 단백질 entity ≥ 3 · full-text `"Fab antibody complex"`
- 정렬 — **release date 오름차순**(오래된 entry부터) → 시간이 지나도 목록이 잘 안 흔들려요
- 사슬 역할 파생 — entity 설명(`pdbx_description`)에 `HEAVY`/`LIGHT` 가 있으면 그대로,
  없으면(`"FAB NC10"` 같은 이름) 사슬 ID 가 `H*`/`L*` 인지로 추정하고,
  그것도 안 되면 **`ab_other_chains`**(= 항체는 맞는데 역할 미분류) 에 남깁니다

In [ ]:
import pathlib

run_tool([PY, SCRIPTS/"fetch_rcsb_ab_snapshot.py",
          "--rows", "12", "--out", "my_run/rcsb_ab_complexes.csv"])

snap_p = pathlib.Path("my_run/rcsb_ab_complexes.csv")
print("\n판정 —", f"스냅샷 생성 완료 ({snap_p})" if snap_p.exists()
      else "스냅샷을 못 만들었어요(네트워크 차단 등) → 아래 절은 커밋된 data/ 로 이어갑니다")

## 3) 내 결과 확인 — 스냅샷 검수 (본문 2.2b)

받아온 표를 그대로 믿으면 안 돼요. **사슬 ID·역할·항원 정체**를 코드로 훑어 손볼 곳을 골라냅니다.
본문이 말한 함정 세 가지가 이 12건 안에 실제로 들어 있어요.

In [ ]:
import pandas as pd
from collections import defaultdict

snap = pd.read_csv(resolve("rcsb_ab_complexes.csv"))

WANT = ["pdb_id", "released", "resolution_A", "heavy_chains", "light_chains",
        "ab_other_chains", "antigen_chains", "antigen_name"]
S = snap.reindex(columns=WANT).copy()          # 없는 컬럼이 있어도 죽지 않게
for col in S.columns:
    if col != "resolution_A":
        S[col] = S[col].fillna("").astype(str)
display(S)

if S["resolution_A"].notna().any():
    r = S["resolution_A"].dropna()
    print("entry %d건 | 해상도(Å) 평균 %.2f · 최고 %.2f · 최저 %.2f | 공개일 %s ~ %s"
          % (len(S), r.mean(), r.min(), r.max(), S.released.min(), S.released.max()))

def ids(v):
    return [x for x in str(v).split(";") if x]

# 함정 ① 항체 사슬 ID 는 규칙이 아니라 관례
odd = S[(S.heavy_chains != "H") | (S.light_chains != "L")]
print("\n① H/L 관례를 안 따르는 entry:",
      ", ".join(f"{t.pdb_id}(H={t.heavy_chains or '-'}, L={t.light_chains or '-'})"
                for t in odd.itertuples()) or "없음")

# 함정 ② 항체인 건 확실한데 heavy/light 로 역할을 못 가른 사슬
unassigned = S[S.ab_other_chains != ""]
print("② 역할 미분류 항체 사슬(ab_other_chains)을 가진 entry:",
      ", ".join(f"{t.pdb_id}({t.ab_other_chains})" for t in unassigned.itertuples()) or "없음")

roles = defaultdict(lambda: defaultdict(list))
for t in S.itertuples():
    for ch in ids(t.heavy_chains):    roles[ch]["항체 heavy"].append(t.pdb_id)
    for ch in ids(t.light_chains):    roles[ch]["항체 light"].append(t.pdb_id)
    for ch in ids(t.ab_other_chains): roles[ch]["항체 역할 미분류"].append(t.pdb_id)
    for ch in ids(t.antigen_chains):  roles[ch]["항원"].append(t.pdb_id)
dual = {k: v for k, v in roles.items() if len(v) > 1}
print(f"   사슬 문자 {len(roles)}개 중 {len(dual)}개가 entry 마다 역할이 달라요.")
for ch in sorted(dual):
    print(f"     '{ch}' — " + " / ".join(f"{role} " + "·".join(sorted(set(e)))
                                         for role, e in dual[ch].items()))

# 함정 ③ full-text 검색이 물어온 노이즈
noise = S[S.antigen_name.str.upper().str.contains("T-CELL|T CELL|RECEPTOR", regex=True)]
print("③ 항원 자리에 항체가 아닌 면역수용체가 앉은 entry:",
      ", ".join(f"{t.pdb_id}({t.antigen_name})" for t in noise.itertuples()) or "없음")

review = sorted(set(odd.pdb_id) | set(unassigned.pdb_id) | set(noise.pdb_id))
print(f"\n판정 — 눈으로 검수해야 할 entry {len(review)}/{len(S)}건 ({100*len(review)/len(S):.0f}%) — "
      + (", ".join(review) or "없음"))
print("      0건이면 그대로 써도 되고, 1건이라도 있으면 chain ID 와 항원 정체를 직접 확인해야 해요.")

have = [f for f, colname in SABDAB_FIELDS.items() if colname and colname in snap.columns]
gap  = [f for f in SABDAB_FIELDS if f not in have]
print(f"\n1절 필드 {len(SABDAB_FIELDS)}개 중 이 스냅샷이 채우는 것 {len(have)}개 — {', '.join(have)}")
print(f"      못 채우는 {len(gap)}개 — {', '.join(gap)}")
print("      이 빈칸이 바로 SAbDab 같은 항체 전용 annotation DB 가 따로 존재하는 이유예요.")

## 4) 레퍼런스 대조 — 커밋된 스냅샷과 비교 (본문 2.2b)

`data/rcsb_ab_complexes.csv` 는 **2026-07-14 에 같은 스크립트로 받아 커밋한 대조군**이에요.
PDB 는 매주 자라니까 내 결과와 100% 같지 않을 수 있어요 — 그 차이 자체가 "DB 는 살아 있다"는 관찰입니다.

In [ ]:
import pandas as pd, pathlib

ref_p, mine_p = pathlib.Path("data/rcsb_ab_complexes.csv"), pathlib.Path("my_run/rcsb_ab_complexes.csv")
if not mine_p.exists():
    print("my_run 스냅샷이 없어 대조를 건너뜁니다 — 위 3절 표는 커밋된 레퍼런스예요.")
elif not ref_p.exists():
    print("레퍼런스가 없어 대조를 건너뜁니다:", ref_p)
else:
    mine, ref = pd.read_csv(mine_p), pd.read_csv(ref_p)
    a, b = set(mine.pdb_id), set(ref.pdb_id)
    print(f"내 결과 {len(a)}개 · 레퍼런스 {len(b)}개 · 공통 {len(a & b)}개")
    print("  내 결과에만 — " + (" · ".join(sorted(a - b)) or "없음"))
    print("  레퍼런스에만 — " + (" · ".join(sorted(b - a)) or "없음"))

    key = [c for c in ["resolution_A", "heavy_chains", "light_chains",
                       "ab_other_chains", "antigen_chains"] if c in mine.columns and c in ref.columns]
    both = mine.merge(ref, on="pdb_id", suffixes=("_mine", "_ref"))
    drift = {c: both.loc[both[f"{c}_mine"].astype(str) != both[f"{c}_ref"].astype(str), "pdb_id"].tolist()
             for c in key}
    drift = {c: v for c, v in drift.items() if v}
    if drift:
        print(f"\n공통 {len(a & b)}건 중 값이 달라진 컬럼")
        display(pd.DataFrame([
            {"컬럼": c, "달라진 entry": f"{len(v)}건",
             "어느 entry 인가": " · ".join(v[:8]) + (f" … 외 {len(v) - 8}개" if len(v) > 8 else "")}
            for c, v in drift.items()]))
    else:
        print(f"  공통 entry 는 {len(key)}개 컬럼이 전부 같아요 (완전 일치).")

    print(f"\n판정 — 공통 {len(a & b)}건의 {len(key)}개 컬럼이 모두 같으면 재현 성공이에요.")
    print("      목록 자체가 달라진 건 오류가 아니라 PDB 가 자란 결과예요 — 스냅샷에는 항상 받은 날짜를 적으세요.")

> 다음 → 본문 [03. 분석 환경 구축](../03_setup/03_setup.md)